In [ ]:
"""
To run this test, download the Taipei MRT dataset folder from 
https://sites.google.com/view/gbatch
and place the 'mrt_data' folder as a top-level subfolder in 
datasets_and_dataloaders. Then run this script.
"""

import os
import sys
import numpy as np
import pandas as pd
import pyscamp

import random
import importlib

current_dir = os.getcwd()
sys.path.append(current_dir)
parent_dir = os.path.dirname(current_dir)
sys.path.append(parent_dir)

from datasets_and_dataloaders.dataloader import load_taipeiMRT


import mplot_python.MINT as M
importlib.reload(M)

from mplot_python.MINT import processAll, nnrobustpca_stable_pcp


In [2]:
data_path = os.path.join(parent_dir, 'datasets_and_dataloaders', 'mrt_data')

df = load_taipeiMRT(data_path=data_path)
print(df.head(30))
print(df.tail())

limit = 5  # threshold for switching to random fill

for column in df.columns:
    if column == "timestamps":
        continue

    series = df[column].copy()
    mask = series.isna()

    # --- Identify NaN runs ---
    group = (mask != mask.shift()).cumsum()
    nan_groups = series[mask].groupby(group[mask])

    # Compute column-wide mean and std for random filling
    mean = np.nanmean(series)
    sigma = np.nanstd(series)
    rand_background = np.random.normal(mean, sigma, len(series))

    for g, idxs in nan_groups.groups.items():
        run_length = len(idxs)
        start = idxs[0]
        end = idxs[-1]

        # --- short run (< limit) -> deterministic interpolation ---
        if run_length < limit:
            print(f"Imputing {run_length} NaNs in column '{column}' at positions {start}-{end} with interpolation.")
            prev_idx = start - 1 if start > 0 else None
            next_idx = end + 1 if end + 1 < len(series) else None

            if prev_idx is not None and next_idx is not None and not np.isnan(series[prev_idx]) and not np.isnan(series[next_idx]):
                interp_vals = np.linspace(series[prev_idx], series[next_idx], run_length + 2)[1:-1]
                series.iloc[idxs] = interp_vals
            else:
                 #  fill with mean if one side missing
                series.iloc[idxs] = mean

        # --- long run (>= limit) -> random Gaussian fill ---
        else:
            print(f"Imputing {run_length} NaNs in column '{column}' at positions {start}-{end} with random Gaussian fill.")

            series.iloc[idxs] = rand_background[idxs]

    df[column] = series

0    2015-11-01 05:00:00
1    2015-11-01 06:00:00
2    2015-11-01 07:00:00
3    2015-11-01 08:00:00
4    2015-11-01 09:00:00
5    2015-11-01 10:00:00
6    2015-11-01 11:00:00
7    2015-11-01 12:00:00
8    2015-11-01 13:00:00
9    2015-11-01 14:00:00
10   2015-11-01 15:00:00
11   2015-11-01 16:00:00
12   2015-11-01 17:00:00
13   2015-11-01 18:00:00
14   2015-11-01 19:00:00
15   2015-11-01 20:00:00
16   2015-11-01 21:00:00
17   2015-11-01 22:00:00
18   2015-11-01 23:00:00
19   2015-11-02 00:00:00
20   2015-11-02 01:00:00
21   2015-11-02 05:00:00
22   2015-11-02 06:00:00
23   2015-11-02 07:00:00
24   2015-11-02 08:00:00
25   2015-11-02 09:00:00
26   2015-11-02 10:00:00
27   2015-11-02 11:00:00
28   2015-11-02 12:00:00
29   2015-11-02 13:00:00
30   2015-11-02 14:00:00
31   2015-11-02 15:00:00
32   2015-11-02 16:00:00
33   2015-11-02 17:00:00
34   2015-11-02 18:00:00
35   2015-11-02 19:00:00
36   2015-11-02 20:00:00
37   2015-11-02 21:00:00
38   2015-11-02 22:00:00
39   2015-11-02 23:00:00


In [ ]:
subsequenceLength = 146

listOfStationsRaw = []
with open(os.path.join(data_path, 'station_name_en.txt'), 'r') as listFile:
    lines = listFile.readlines()
    for line in lines:
        line = line.strip()
        listOfStationsRaw.append(line + " enter")

listOfStationsRaw[0] = "Songshan Airport enter"
col_list = listOfStationsRaw[0:15] + listOfStationsRaw[17:]
exclude = []

print("filtering sensors...")
for idx, sensor_name in enumerate(col_list):

    print("Processing sensor:", sensor_name)
    series = df[sensor_name].to_numpy().astype(np.float32)    

    mplot = pyscamp.abjoin_matrix(
        series, series, subsequenceLength,
        mheight=258, mwidth=258, threshold=-1
    )

    if np.isnan(mplot).any():
        print(sensor_name + " excluded due to NaNs in MATRIX PROFILE")
        exclude.append(idx)
        
col_list = [col for i, col in enumerate(col_list) if i not in exclude]



results =  processAll(col_list, df, subsequenceLength, Mheight = 258, Mwidth = 258, name = "Taipei")

In [8]:
from mplot_python.MINT import plotMatrixRaw
import numpy as np

A,B,C = results.low_rank_factors

A = np.log(np.abs(A))
B = np.log(np.abs(B))
C = np.log(np.abs(C))

color_map = "viridis"

plotMatrixRaw(A, "Taipei_A", "Taipei", colormap = color_map)
plotMatrixRaw(B, "Taipei_B", "Taipei", colormap = color_map)
plotMatrixRaw(C, "Taipei_C", "Taipei", colormap = color_map)

NameError: name 'results' is not defined

In [ ]:
component_indices = range(C.shape[1])
s = 5
for i in component_indices:
    print(f"Component {i}")
    component = C[:, i]
    component_abs = np.abs(component)

    # Top-s indices in descending order
    ind = np.argpartition(component_abs, -s)[-s:]
    top_s_indices = ind[np.argsort(component_abs[ind])]
    top_s_indices = np.flip(top_s_indices)
    
    for j in top_s_indices:
        print(j, pd.to_datetime(df.loc[0, "timestamps"]) + pd.Timedelta(hours = (len(df) / 258) * j))

In [4]:
from mplot_python.co_clustering_trial import create_processed_dataframe

b_prop = 0.15
num_of_windows = 12
window_size = 146


processed_dataframe, chosen_intervals, completely_random, mostly_random, mostly_normal = create_processed_dataframe(df, col_list, b_prop, num_of_windows, window_size)



In [5]:
print(len(processed_dataframe))

10865


In [6]:
print(chosen_intervals)
print(completely_random)
print(mostly_random)
print(mostly_normal)

[[6570, 6715], [438, 583], [8906, 9051], [5548, 5693], [9052, 9197], [10220, 10365], [0, 145], [7738, 7883], [2482, 2627], [730, 875], [7300, 7445], [292, 437]]
['Xiaobitan enter', 'Dingpu enter', 'Fuzhong enter', 'Gongguan enter', 'Fu Jen University enter', 'Dazhi enter', 'Far Eastern Hospital enter', 'Huzhou enter', 'Wende enter', 'Zhongyi enter', 'Beitou enter', 'Zhongxiao Fuxing enter', 'Liuzhangli enter', 'Nanjing Sanmin enter', 'Zhongshan Junior High School enter', 'Tamsui enter']
['Jingmei enter', 'Donghu enter', 'Guting enter', 'Houshanpi enter', 'Chiang Kai-shek Memorial Hall enter', 'Xianse Temple enter', 'Taipower Building enter', 'Shipai enter', 'Xinhai enter', 'Kunyang enter', 'Dingxi enter', 'Linguang enter', 'Fuxinggang enter', 'Xinzhuang enter', 'Songshan enter', 'Dapinglin enter']
['Yongan Market enter', 'Banqiao enter', 'Taipei 101/World Trade Center enter', 'Touqianzhuang enter', 'Nangang Exhibition Center enter', 'Zhongshan enter', 'Taipei Arena enter', 'Xindian ent

In [ ]:
from mplot_python.MINT import plotMatrixRaw
color_map = "viridis"

# NOTE: group membership is random per run; OPSD hardcoded "Germany" after
# inspecting the printout above. Taipei station names can't be known in
# advance, so take the first mostly_random sensor programmatically.
rand = processed_dataframe[mostly_random[0]].to_numpy()
rand_mplot = pyscamp.abjoin_matrix(
                np.copy(rand),  # Convert to Python list
                np.copy(rand),  # Convert to Python list
                subsequenceLength, 
                mheight=258, 
                mwidth=258, 
                threshold=-1
            )

plotMatrixRaw(rand_mplot, "Taipei_rand_M", "Taipei", colormap = color_map)




In [13]:
print(num_iter)

924


In [14]:
rand_sL, rand_sS, num_iter = nnrobustpca_stable_pcp(rand_mplot, max_iter = 5000)

plotMatrixRaw(rand_sL, "Taipei_rand_sL", "Taipei", colormap = color_map)
plotMatrixRaw(rand_sS, "Taipei_rand_sS", "Taipei", colormap = color_map)

In [ ]:
# First mostly_normal sensor (OPSD hardcoded "Portugal"; see note above)
norm = processed_dataframe[mostly_normal[0]].to_numpy()
norm_mplot = pyscamp.abjoin_matrix(
                np.copy(norm),  # Convert to Python list
                np.copy(norm),  # Convert to Python list
                subsequenceLength, 
                mheight=258, 
                mwidth=258, 
                threshold=-1
            )

plotMatrixRaw(norm_mplot, "Taipei_norm_M", "Taipei", colormap = color_map)




In [16]:
norm_sL, norm_sS, num_iter = nnrobustpca_stable_pcp(norm_mplot, max_iter = 5000)

plotMatrixRaw(norm_sL, "Taipei_norm_sL", "Taipei", colormap = color_map)
plotMatrixRaw(norm_sS, "Taipei_norm_sS", "Taipei", colormap = color_map)